<a href="https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Openenv_wordle_grpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# <img width="35" height="35" alt="image" src="https://github.com/user-attachments/assets/2700a971-e5d6-4036-b03f-2f89c9791609" /> OpenEnv：代理执行环境
我们正在使用新的 [OpenEnv](https://github.com/meta-pytorch/OpenEnv) 库，该库拥有超过 2000 多个 RL 环境！

要运行此程序，请在 A100 Google Colab Pro 实例上按“*运行时*”并按“*全部运行*”！
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> 如果您需要帮助，请加入 Discord + ⭐ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</a> </i> ⭐
</div>

要在本地设备上安装 Unsloth，请按照 [our guide](https://unsloth.ai/docs/get-started/install) 操作。

# 目标：让Qwen3-4B用强化学习玩游戏

我们的目标是让 OpenAI 的开放模型 Qwen3-4B 通过强化学习来玩填字游戏。我们希望模型设计一个玩单词的策略，我们将运行这个策略，直到我们赢或输。

<img width="270" height="265" alt="image" src="https://github.com/user-attachments/assets/1a51a172-ff1a-40a2-8bee-2746f5017aa0" />

# 安装
我们将使用 [Unsloth](https://github.com/unslothai/unsloth) 在 Qwen3-4B 上进行强化学习，并使用 [OpenEnv](https://github.com/meta-pytorch/OpenEnv) 进行环境交互。 Unsloth 节省了 70% 的 VRAM 使用量，并使强化学习速度提高 2 到 6 倍！

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356# 子目录=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 "tokenizers>=0.22.0,<=0.23.0" trl==0.29.1 unsloth unsloth_zoo

然后我们将从源安装 [OpenEnv](https://github.com/meta-pytorch/OpenEnv)：

In [ ]:
%%capture
!pip install -qqq fastapi uvicorn requests open_spiel --prefer-binary
!pip install "openenv-core[core]>=0.2.1"
!git clone https://github.com/meta-pytorch/OpenEnv.git > /dev/null 2>&1
%cd OpenEnv
import subprocess, sys, os
from pathlib import Path
sys.path.insert(0, './envs')  # 为 textarena_env 模块添加 OpenEnv 环境
sys.path.insert(0, './src')
working_directory = str(Path.cwd().absolute())

我们将加载 Qwen3-4B 并设置一些参数：
* `max_seq_length = 4096` 模型的最大上下文长度。增加它会使用更多内存。
* `lora_rank = 32` 这个数字越大，RL 过程越智能，但速度越慢，占用的内存也越多。

In [ ]:
import os
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # 可以增加更长的 RL 输出
lora_rank = 32       # 排名越高=越聪明，但速度越慢
model, processor = FastLanguageModel.from_pretrained(
    # model_name = "unsloth/gemma-3-1b-it",
    model_name = 'unsloth/Qwen3-4B',
    load_in_4bit = False,
    max_seq_length = max_seq_length,
    fast_inference = True,
)
tokenizer = getattr(processor, 'tokenizer', processor)

[unsloth.import_fixes|INFO]Unsloth: ROCm detection did not match any known hints.
[unsloth.import_fixes|INFO]Unsloth: Patching protobuf.MessageFactory.GetPrototype
[unsloth.import_fixes|INFO]Unsloth: fbgemm_gpu_genai==1.5.0+cu130 detected.
[unsloth.import_fixes|INFO]Unsloth: torch==2.10.0+cu130 and torchvision==0.25.0+cu130 are compatible.


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


为了进行高效的强化学习，我们将使用 [LoRA](https://arxiv.org/abs/2106.09685)，它允许我们仅向模型添加 1 到 5% 的额外权重以进行微调。这使我们能够节省 60% 以上的内存使用量，同时保持良好的准确性。阅读 Unsloth 的 [Reinforcement Learning Guide](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 了解更多详细信息。

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # 选择任意大于 0 的数字！建议 8、16、32、64、128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 加快训练速度
    use_gradient_checkpointing = "unsloth", # 减少内存使用
    random_state = 3407,
)

## 初始化环境

让我们首先设置训练期间使用的环境。  
对于此任务，我们将依赖 **OpenEnv** 中的 **TextArena** 环境，它公开了熟悉的 Gymnasium 风格的 API（`reset()`、`step()` 等）来简化交互。

在此示例中，我们将连接到位于 [openenv/wordle](https://huggingface.co/spaces/openenv/wordle) 的托管环境。  
对于生产使用或自定义配置，我们**强烈建议**通过 Docker 在本地运行环境。 Hub 上的托管版本目前的并发支持有限，因此在这些情况下，将空间复制到您自己的帐户是首选方法。

有关更多信息，请参阅 [TRL-OpenEnv documentation](https://huggingface.co/docs/trl/main/en/openenv)。

In [ ]:
# notebook_login() # 如果要推送到 HF Hub，请取消注释
# 从huggingface_hub导入notebook_login
# 笔记本_登录()

In [ ]:
from textarena_env import TextArenaEnv

wordle_url = "https://openenv-wordle.hf.space" # 复制空间并更新它！
env = TextArenaEnv(base_url = wordle_url).sync()  # 需要 .sync()：OpenEnv 客户端默认是异步的
# wordle_url =“openenv/wordle”
# env = TextArenaEnv.from_hub(repo_id = wordle_url)

## 带有助手的推出功能

**rollout 函数**定义了 GRPO 训练期间代理如何与环境交互。
它负责生成模型完成、收集反馈（奖励）并返回优化所需的所有信息。

在此设置中：
- 在每个训练步骤中，**GRPOTrainer** 自动调用该函数。  
- 它使用训练器的内置“generate_rollout_completions()”方法，在并置模式下使用 vLLM 进行高效生成。
- 每次推出都代表一个完整的交互循环。该模型进行猜测、接收来自 Wordle 的反馈，并根据奖励信号进行更新。
- **`env_mask`** 跟踪哪些标记是模型生成的还是环境生成的，确保只有模型标记才会导致训练损失。

奖励跟踪代理绩效的不同方面。辅助函数（如“rollout_once”）处理一轮交互，保持主“rollout_func”的干净和模块化。

这种模块化方法使 GRPO 能够通过强化学习有效地采样、评估和改进模型的猜测策略。

首先，我们定义“system_prompt”，它作为具有策略推理和结构化响应的专家 Wordle 求解器来指导模型的行为。

In [ ]:
# @title 系统提示（点击展开）
system_prompt = """
You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

# # 游戏规则

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN or G: Letter is correct and in the correct position
   - YELLOW or Y: Letter is in the word but in the wrong position
   - GRAY or X: Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed

# # 响应格式

Format your response as follows:
<reasoning>
Briefly analyze the feedback from previous guesses, eliminating letters and finding possible words.
</reasoning>
[guess]

# # 重要限制

- Use lowercase only
- One guess per response
- Must be exactly 5 letters
- Must be a real English word from standard dictionaries
- Never repeat a previous guess

# # 你的目标

Solve the Wordle in as few guesses as possible by strategically using feedback to eliminate impossible words and narrow down the solution space efficiently."""

现在，让我们定义“rollout_func”：

该函数协调模型和 Wordle 环境之间的交互。对于批次中的每个提示，它都会运行情节交互，收集奖励和模型输出以进行 GRPO 优化。

In [ ]:
max_new_tokens = 256
max_turns = 6

def rollout_func(prompts, trainer):
    """
    Rollout function for GRPO training with environment interaction.

    This function is called by GRPOTrainer to generate completions and compute rewards.
    It uses trainer.generate_rollout_completions() for inference.

    Args:
        prompts: List of prompts to generate from
        trainer: GRPOTrainer instance containing context and configuration

    Returns:
        Dictionary with prompt_ids, completion_ids, logprobs, env_mask, and reward signals
    """
    episode_prompt_ids = []
    episode_completion_ids = []
    episode_logprobs = []
    episode_env_masks = []
    correctness_rewards = []
    position_rewards = []
    format_rewards = []

    for prompt_text in prompts:
        episode = rollout_once(
            trainer = trainer,
            env = env,
            tokenizer = tokenizer,
            dataset_prompt = prompt_text,
            system_prompt = system_prompt,
            max_turns = max_turns,
            max_new_tokens = max_new_tokens,
        )
        episode_prompt_ids.append(episode["prompt_ids"])
        episode_completion_ids.append(episode["completion_ids"])
        episode_logprobs.append(episode["logprobs"])
        episode_env_masks.append(episode["env_mask"])
        correctness_rewards.append(episode["correct_reward"])
        position_rewards.append(episode["position_reward"])
        format_rewards.append(compute_format_reward(episode["model_outputs"]))

    return {
        "prompt_ids": episode_prompt_ids,
        "completion_ids": episode_completion_ids,
        "logprobs": episode_logprobs,
        "env_mask": episode_env_masks,
        "correct_reward": correctness_rewards,
        "position_reward": position_rewards,
        "format_reward": format_rewards,
    }

### 定义`rollout_once`

“rollout_once”函数使用训练器的生成方法在模型和 Wordle 环境之间运行**一个完整的交互循环**。  
它执行一个迷你的游戏情节，从生成猜测到接收和处理反馈。

以下是分步说明：

1. **环境重置：** 开始新的游戏会话并初始化观察。  
2. **提示构建：** 将系统提示、当前状态和用户消息组合起来形成模型输入。  
3. **生成：** 使用 `trl.experimental.openenv.generate_rollout_completions()` 有效地生成模型的猜测。  
4. **反馈提取：** 使用 `extract_guess()` 和 `extract_wordle_feedback()` 等帮助器解析环境的响应。  
5. **奖励计算：** 根据正确性、绿/黄反馈和重复惩罚来计算奖励。
6. **返回结构化推出数据：** 包括提示/完成 ID、logprobs、`env_mask` 和所有计算的奖励组件。

**重要：`env_mask`机制**

在 Wordle 等多轮环境中，完成包括：
- **模型生成的标记**（猜测）：这些应该会导致训练期间的损失。
- **环境反馈代币**（游戏响应）：这些不应导致损失。

`env_mask` 是一个 1 和 0 的列表，标记哪些标记是模型生成的 (`1`) 与环境生成的 (`0`)。  
GRPOTrainer 使用此掩码从损失计算中排除环境标记，确保模型仅从其自己的输出中学习。

这种模块化设计确保每个情节都可以独立处理，同时仍然为**GRPO 训练循环**提供丰富的反馈。

In [ ]:
import re
from textarena_env import TextArenaAction
from textarena_env.rewards import extract_feedback_counts, extract_guess, extract_wordle_feedback
from trl.experimental.openenv import generate_rollout_completions

def rollout_once(trainer, env, tokenizer, dataset_prompt, system_prompt, max_turns, max_new_tokens):
    result = env.reset()
    observation = result.observation

    prompt_ids = []
    completion_ids = []
    logprobs = []
    env_mask = []  # 1 表示模型生成的令牌，0 表示环境令牌
    model_outputs = []
    raw_rewards = []
    position_scores = []
    correct_scores = []
    prev_env_output_len = 0  # 每转仅添加新部分的轨道长度

    accumulated_messages: list[dict[str, str]] = [{"role": "system", "content": system_prompt}]
    # 构建初始提示（仅一次，在开始时）
    # 初始环境消息包含在提示中，而不是完成中
    base_prompt = observation.prompt or dataset_prompt
    initial_user_prompt = make_user_prompt(observation.messages)
    # 跟踪初始环境输出长度，这样我们就不会再次添加它
    initial_env_output = format_history(observation.messages) if observation.messages else ""
    prev_env_output_len = len(initial_env_output)
    initial_messages = accumulated_messages + [{"role": "user", "content": initial_user_prompt}]
    initial_prompt_text = tokenizer.apply_chat_template(
        initial_messages,
        add_generation_prompt = True,
        tokenize = False,
        enable_thinking = False,
    )
    # 将初始提示标记一次 - 这是整个剧集的基本提示。
    # GRPO 期望每集有一对提示完成对，其中：
    # - Prompt_ids = 初始/基本提示（模型在剧集开始时看到的内容）
    # -completion_ids = 所有模型响应 + 所有回合连接的环境反馈
    # 注意：每轮生成时实际使用的提示较长（包括对话历史记录），
    # 但我们在这里只计算初始提示标记。
    initial_prompt_ids = tokenizer.encode(initial_prompt_text, add_special_tokens = False)
    prompt_ids.extend(initial_prompt_ids)

    for _turn in range(max_turns):
        if result.done:
            break

        base_prompt = observation.prompt or dataset_prompt
        user_prompt = make_user_prompt(observation.messages)
        messages = accumulated_messages + [{"role": "user", "content": user_prompt}]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True,
            tokenize = False,
            enable_thinking = False,
        )

        rollout_outputs = generate_rollout_completions(
            trainer, [prompt_text], generation_overrides = {"max_tokens": max_new_tokens}
        )[0]
        # 添加模型生成的完成标记和带有换行符的日志概率以提高可读性
        newline_tokens = tokenizer.encode("\n", add_special_tokens = False)
        completion_ids.extend(newline_tokens)  # 猜测前换行
        logprobs.extend([0.0] * len(newline_tokens))
        env_mask.extend([0] * len(newline_tokens))  # 合成分隔符，不是模型生成的

        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])
        env_mask.extend([1] * len(rollout_outputs["completion_ids"]))  # 模型生成的令牌

        completion_ids.extend(newline_tokens)  # 猜测后换行
        logprobs.extend([0.0] * len(newline_tokens))
        env_mask.extend([0] * len(newline_tokens))  # 合成分隔符，不是模型生成的
        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens = True
        )
        guess = extract_guess(completion_text)
        model_outputs.append(completion_text.strip())  # 存储原始模型输出以获得格式奖励

        result = env.step(TextArenaAction(message = guess))

        raw_rewards.append(float(result.reward or 0.0))
        observation = result.observation
        correct_score = float(result.reward or 0.0)
        feedback = extract_wordle_feedback(observation)

        full_env_output = format_history(observation.messages) if observation.messages else ""
        new_env_output = full_env_output[prev_env_output_len:].lstrip("\n")
        prev_env_output_len = len(full_env_output)

        if new_env_output:
            env_output_tokens = tokenizer.encode(new_env_output, add_special_tokens = False)
            completion_ids.extend(env_output_tokens)  # 添加到completion_ids
            logprobs.extend([0.0] * len(env_output_tokens))  # 占位符（通过 env_mask = 0 忽略）
            env_mask.extend([0] * len(env_output_tokens))  # 环境代币 - 避免损失
            completion_with_env = completion_text + "\n" + new_env_output
        else:
            completion_with_env = completion_text

        accumulated_messages.append({"role": "user", "content": user_prompt})
        accumulated_messages.append({"role": "assistant", "content": completion_with_env})

        if not feedback:
            position_score = 0.0
        else:
            green_count, yellow_count = extract_feedback_counts(feedback)
            position_score = (green_count + 0.5 * yellow_count) / 5.0

        position_scores.append(position_score)
        correct_scores.append(correct_score)

    # 使用最终正确的奖励（赢/输最后是二元的）
    correct_reward_value = correct_scores[-1] if correct_scores else (raw_rewards[-1] if raw_rewards else 0.0)

    # 将奖励定位为塑造信号：
    # - 如果模型获胜：position_reward = 1.0（快速获胜不会受到惩罚）
    # - 如果模型失败：position_reward = 最后一次尝试（最终的位置）
    if correct_reward_value >= 1.0:
        final_position_reward = 1.0
    else:
        final_position_reward = position_scores[-1] if position_scores else 0.0

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "env_mask": env_mask,
        "raw_rewards": raw_rewards,
        "correct_reward": correct_reward_value,
        "position_reward": final_position_reward,
        "model_outputs": model_outputs,
    }

### 辅助函数

支持“rollout_once”中使用的实用程序：

- **`make_user_prompt`**：结合对话历史记录构建用户提示。
- **`format_history`**：格式化对话日志以获得一致的上下文。

In [ ]:
# @title Helpers 定义（点击展开）
def format_history(messages) -> str:
    lines = []
    for message in messages:
        tag = message.category or "MESSAGE"
        content = message.content.strip()
        if not content:
            continue
        lines.append(f"[{tag}] {content}")
    return "\n".join(lines)


def make_user_prompt(messages) -> str:
    history = format_history(messages)
    # 仅将消息用于对话历史记录 - 提示已包含在第一条消息中
    history_section = history if history else "[PROMPT] Awaiting first feedback."
    return f"Conversation so far:\n{history_section}\n\nReply with your next guess enclosed in square brackets."

## 定义奖励函数

为了指导代理的学习过程，我们定义了简单的奖励函数，将环境的反馈映射为数字信号。  
每个功能对应于 **Wordle** 游戏的特定方面：

- ✅ **`reward_ Correct`**：当模型猜出正确的单词时奖励模型（二进制：0 或 1）。  
- 🎯 **`reward_position`**：根据信件反馈奖励进度。绿色字母值 1.0，黄色字母值 0.5，标准化为 5。如果模型获胜，则设置为 1.0。
- 📝 **`reward_format_strict`**：奖励正确的输出格式`[xxxxx]`。返回所有回合中格式正确的输出的比例。

这些函数返回 **GRPOTrainer** 在优化期间使用的浮点值列表。  
通过将它们结合起来，模型学会在猜测策略中平衡正确性、信息收集和正确的格式。

In [ ]:
os.environ['STEP_NUM'] = '0'
def reward_correct(completions, **kwargs):
    """Reward from environment (correct answer)."""
    rewards = kwargs.get("correct_reward") if kwargs else None
    if rewards is None:
        return [0.0 for _ in completions]
    os.environ['STEP_NUM'] = str(int(os.environ['STEP_NUM'])+1)
    if int(os.environ['STEP_NUM'])%10==0:
        print(f'{completions=}')
    return [float(r) for r in rewards]


def reward_position(completions, **kwargs):
    """Position reward: green worth 1.0, yellow worth 0.5, normalized by 5."""
    rewards = kwargs.get("position_reward") if kwargs else None
    if rewards is None:
        return [0.0 for _ in completions]
    return [float(r) for r in rewards]


def compute_format_reward(model_outputs):
    """Compute format reward from a list of model outputs (one per turn).

    Each output should have a <reasoning> block and a [guess].
    Returns proportion of correctly formatted outputs.
    """
    if not model_outputs:
        return 0.0

    exact_pattern = re.compile(r"^.*<reasoning>.*?</reasoning>\s*\[[A-Za-z]{5}\]\s*$", re.DOTALL)
    correct_count = sum(1 for output in model_outputs if exact_pattern.match(output))

    return correct_count / len(model_outputs)


def reward_format_strict(completions, **kwargs):
    """Format reward - pre-computed in rollout_func."""
    rewards = kwargs.get("format_reward") if kwargs else None
    if rewards is None:
        return [0.0 for _ in completions]
    return [float(r) for r in rewards]

## 创建数据集

我们创建一个带有重复提示的数据集来控制训练集的数量。  
数据集中的每个条目都会在训练期间触发一个 rollout 事件。 “dataset_prompt”在每个游戏开始之前向模型提供初始指令。

In [ ]:
from datasets import Dataset

dataset_size = 3000
dataset_prompt = "Play Wordle like an expert."

dataset = Dataset.from_dict({"prompt": [dataset_prompt] * dataset_size})

## 设置 GRPO 配置

接下来，我们定义 **GRPOConfig**，它控制所有关键训练参数。  
此配置指定模型如何与 **vLLM** 交互、管理内存和记录结果。

In [ ]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    # 培训计划/优化
    num_train_epochs = 1,                     # 完整数据集传递次数
    learning_rate = 1e-5,                     # 优化器的学习率
    gradient_accumulation_steps = 4,         # 通过多个步骤累积梯度
    per_device_train_batch_size = 1,          # 每个 GPU 的批量大小（一起处理的提示数量）
    warmup_steps = 20,                        # 学习率预热的步骤
    optim = "adamw_8bit",                      # 优化器
    max_grad_norm = 1.0,                        # 剪辑渐变以防止爆炸

    # GRPO配置
    num_generations = 4,                      # 每个提示的推出次数（用于减少方差）
    max_completion_length = 1024,               # 完整剧集长度，而非每回合
    log_completions = False,                  # 记录完成情况以进行调试

    # 记录/报告
    output_dir = 'outputs',                  # 检查点和日志的目录
    report_to = "trackio",                      # 实验跟踪工具（与 HF Spaces 集成）
    trackio_space_id = 'outputs',            # 保存实验跟踪的 HF 空间
    logging_steps = 10,                        # 每 N 步记录一次指标
    # save_steps = 10, # 保存检查点的时间间隔

    top_p = 0.95,
    top_k = 50,

    gradient_checkpointing = True,            # 启用激活重新计算以节省内存
    beta = 0.02
)

## 创建“GRPOTrainer”并开始训练

现在我们初始化“GRPOTrainer”，它管理整个强化学习循环。

它采用之前定义的模型、分词器、奖励函数、推出函数和数据集。  
训练器协调模型与环境之间的交互，应用奖励信号并更新策略。

最后，我们调用trainer.train()来开始微调过程，让模型通过反馈和迭代来学习玩Wordle。

In [ ]:
import sys
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        reward_correct,
        reward_position,
        reward_format_strict,
    ],
    train_dataset = dataset,
    args = grpo_config,
    rollout_func = rollout_func,
)

In [ ]:
type(trainer.processing_class)

在训练前显示内存统计数据

In [ ]:
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

并训练！

In [ ]:
trainer_stats = trainer.train()

显示训练后的记忆统计数据

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

In [ ]:
env.close()
trainer.save_model(grpo_config.output_dir)
# trainer.push_to_hub() # 取消注释以推送到 HF Hub

## 推理

现在，让我们使用内存中已有的模型运行 **推理** 来测试我们的微调模型。  
我们使用 FastLanguageModel.for_inference(model) 将模型切换到推理模式并玩 Wordle 游戏。

In [ ]:
# 使用内存中已有的模型（无需从磁盘重新加载）
FastLanguageModel.for_inference(model)

现在我们已经加载了微调的模型，我们可以开始玩 Wordle。  
为了使这更容易，我们将定义一个可重用的函数，以便我们可以进行多轮比赛。  
该函数实现了我们之前探索的相同逻辑。

In [ ]:
MAX_TURNS = 6

def play_wordle(env, model, tokenizer):
    result = env.reset()
    observation = result.observation

    print("📜 Initial Prompt:\n" + observation.prompt)

    for turn in range(MAX_TURNS):
        if result.done:
            break

        user_prompt = make_user_prompt(observation.messages)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True,
            tokenize = False,
            enable_thinking = False,
        )

        model_inputs = tokenizer([prompt_text], return_tensors = "pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens = 512
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

        # 解码并提取模型响应
        generated_text = tokenizer.decode(output_ids, skip_special_tokens = True)
        guess = extract_guess(generated_text)

        print(f"\n🎯 Turn {turn}: model replied with -> {generated_text}")
        print(f"   Parsed guess: {guess}")

        result = env.step(TextArenaAction(message = guess))
        observation = result.observation

        print("反馈信息：")
        for message in observation.messages:
            print(f"     [{message.category}] {message.content}")

    print("\n✅ 游戏结束")
    print(f"   Reward: {result.reward}")
    print(f"   Done: {result.done}")

我们来玩游戏吧！

In [ ]:
# 重新打开环境进行推理（训练环境已关闭）
inference_env = TextArenaEnv(base_url = wordle_url).sync()
try:
    play_wordle(inference_env, model, tokenizer)
finally:
    inference_env.close()

我们就完成了！如果您对 Unsloth 有任何疑问，我们有 [Discord](https://discord.gg/unsloth) 频道！如果您发现任何错误或想要随时更新最新的 LLM 内容，或需要帮助、加入项目等，请随时加入我们的 Discord！

其他一些资源：
1. 希望在本地使用 Unsloth？请阅读我们的 [Installation Guide](https://unsloth.ai/docs/get-started/install)，了解有关在 Windows、Docker、AMD、Intel GPU 上安装 Unsloth 的详细信息。
2. 了解如何使用我们的 [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide) 进行强化学习。
3. 阅读我们的指南和笔记本以了解 [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) 和 [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) 模型支持。
4. 浏览我们的 [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) 以查找每个型号的专用指南。
5. 需要推理方面的帮助吗？请阅读我们的 [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment)，了解有关使用 vLLM、llama.cpp、Ollama 等的详细信息。

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  如果您需要帮助，请加入 Discord + ⭐️ <i>在 <a href="https://github.com/unslothai/unsloth">Github</a> 上为我们加注星标</i> ⭐️

  此笔记本和所有 Unsloth 笔记本均已获得许可 [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>